In [38]:
import pandas as pd
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec


nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\AKHIL\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\AKHIL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [39]:
raw_df = pd.read_csv("cleaned_data1.csv")
raw_df

,text,label
0,ad sales boost time warner profit quarterly pr...,business
1,dollar gains on greenspan speech the dollar ha...,business
2,yukos unit buyer faces loan claim the owners o...,business
3,high fuel prices hit bas profits british airwa...,business
4,pernod takeover talk lifts domecq shares in uk...,business
...,...,...
2129,last star wars not for children the sixth and ...,entertainment
2130,french honour for director parker british film...,entertainment
2131,robots march to us cinema summit animated movi...,entertainment
2132,hobbit picture four years away lord of the rin...,entertainment


In [40]:
lemmatizer = WordNetLemmatizer()

In [41]:
import nltk
from nltk.corpus import stopwords


In [42]:
stop_words = set(stopwords.words('english'))

# Function to preprocess sentences
def preprocess(sentences):
    processed = []
    for sentence in sentences:
        # Tokenize the sentence into words
        tokens = word_tokenize(sentence.lower())
        # Remove stopwords and non-alphabetic tokens
        filtered_tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
        processed.append(filtered_tokens)
    return processed

In [43]:
# Function to convert nltk pos tags to wordnet tags
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return None

In [44]:
def lemmatize_text(text):
    word_tokens = word_tokenize(text)
    pos_tags = nltk.pos_tag(word_tokens)
    lemmatized_words = []
    for word, tag in pos_tags:
        wn_tag = get_wordnet_pos(tag)
        if wn_tag:
            lemmatized_words.append(lemmatizer.lemmatize(word, pos=wn_tag))
        else:
            lemmatized_words.append(lemmatizer.lemmatize(word))
    return ' '.join(lemmatized_words)

In [45]:
raw_df['lemmatized_text'] = raw_df['text'].apply(lemmatize_text)



In [46]:
raw_df['tokenized_text'] = raw_df['lemmatized_text'].apply(word_tokenize)

# Train the Word2Vec model
word2vec_model = Word2Vec(
    sentences=raw_df['tokenized_text'].tolist(),  # input list of tokenized sentences
    vector_size=100,  # size of the word vectors
    window=5,  # context window size
    min_count=1,  # ignore words with frequency lower than this
    workers=4,  # number of CPU cores to use
)

# Save the model
word2vec_model.save("word2vec_model.model")

# Example: Get the vector for a specific word (e.g., "profit")
word_vector = word2vec_model.wv['profit']
print("Word vector for 'profit':",len( word_vector))

# Example: Get the most similar words to a given word
# similar_words = word2vec_model.wv.most_similar('profit')
# print("Words most similar to 'profit':", similar_words)

Word vector for 'profit': 100


In [47]:
def get_avg_word2vec(tokens, model, vector_size):
    # Get word vectors for the tokens that exist in the model vocabulary
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if valid_vectors:
        return sum(valid_vectors) / len(valid_vectors)
    else:
        return [0] * vector_size

In [69]:
get_avg_word2vec(raw_df["tokenized_text"][0],word2vec_model,100)

array([-0.2714969 ,  0.21351828,  0.18590179,  0.03456157, -0.064987  ,
       -0.42527488,  0.61923724,  0.87438285, -0.25020415, -0.4688406 ,
        0.3142239 , -0.2211582 , -0.00733932, -0.04988478, -0.02963587,
       -0.4968819 ,  0.45003554,  0.08715715, -0.28746834, -0.7698864 ,
        0.16920276, -0.03449384,  0.5935938 , -0.43650886,  0.27987507,
        0.31762415, -0.22680344,  0.14571548, -0.38458756, -0.0584722 ,
        0.3343711 ,  0.16045621, -0.07559014, -0.42586562, -0.12861727,
        0.46852216,  0.07756994, -0.25716293, -0.17975003, -0.30735913,
        0.06029849, -0.19137463, -0.5360728 , -0.25195423,  0.28030062,
       -0.37293196, -0.26631665, -0.32241   ,  0.01363518, -0.11971287,
       -0.02618077, -0.14167257, -0.14694546, -0.18264987,  0.12121996,
       -0.32015756,  0.32478318, -0.01457889, -0.63595515,  0.01025626,
        0.08214388, -0.48717928,  0.86500734, -0.6405444 , -0.581412  ,
        0.40864977, -0.02194981,  0.43701074, -0.6662157 , -0.01

In [70]:
# Apply the function to create feature vectors
vector_size = 100  # Same size as used in Word2Vec training
raw_df['feature_vector'] = raw_df['tokenized_text'].apply(lambda x: get_avg_word2vec(x, word2vec_model, vector_size))

In [71]:
raw_df["feature_vector"][0]

array([-0.2714969 ,  0.21351828,  0.18590179,  0.03456157, -0.064987  ,
       -0.42527488,  0.61923724,  0.87438285, -0.25020415, -0.4688406 ,
        0.3142239 , -0.2211582 , -0.00733932, -0.04988478, -0.02963587,
       -0.4968819 ,  0.45003554,  0.08715715, -0.28746834, -0.7698864 ,
        0.16920276, -0.03449384,  0.5935938 , -0.43650886,  0.27987507,
        0.31762415, -0.22680344,  0.14571548, -0.38458756, -0.0584722 ,
        0.3343711 ,  0.16045621, -0.07559014, -0.42586562, -0.12861727,
        0.46852216,  0.07756994, -0.25716293, -0.17975003, -0.30735913,
        0.06029849, -0.19137463, -0.5360728 , -0.25195423,  0.28030062,
       -0.37293196, -0.26631665, -0.32241   ,  0.01363518, -0.11971287,
       -0.02618077, -0.14167257, -0.14694546, -0.18264987,  0.12121996,
       -0.32015756,  0.32478318, -0.01457889, -0.63595515,  0.01025626,
        0.08214388, -0.48717928,  0.86500734, -0.6405444 , -0.581412  ,
        0.40864977, -0.02194981,  0.43701074, -0.6662157 , -0.01

In [72]:
# Save DataFrame to a CSV file
raw_df.to_csv('dataset.csv', index=False)

In [21]:
import numpy as np
import gensim
from sklearn.preprocessing import LabelEncoder

# Load the pre-trained Word2Vec model
word2vec_model_path = 'word2vec_model.model'  # Update with the actual path
word2vec = gensim.models.Word2Vec.load(word2vec_model_path)

# Function to convert text into a feature vector using Word2Vec
def text_to_feature_vector(text):
    words = text.split()  # Simple tokenization, you may use a better tokenizer
    word_vectors = []

    for word in words:
        if word in word2vec.wv:  # Check if the word exists in the Word2Vec vocabulary
            word_vectors.append(word2vec.wv[word])
    
    if len(word_vectors) > 0:
        # Return the average of word vectors for all words in the text
        return np.mean(word_vectors, axis=0)
    else:
        # If none of the words are in the Word2Vec model, return a zero vector
        return np.zeros(word2vec.vector_size)

# Function to predict the label for user-entered text
def predict_label(user_text):
    # Convert user-entered text to a feature vector using Word2Vec
    feature_vector = text_to_feature_vector(user_text)

    # Reshape feature_vector for prediction (1 sample with feature_vector length)
    feature_vector = feature_vector.reshape(1, -1)

    # Make prediction using the trained XGBoost classifier
    predicted_label_index = xgb_clf.predict(feature_vector)[0]

    # Get the actual label name from the encoded label
    predicted_label = label_encoder.inverse_transform([predicted_label_index])[0]

    return predicted_label

# Example usage
user_text = input("Enter some text: ")
predicted_label = predict_label(user_text)
print(f"The predicted label is: {predicted_label}")


Enter some text: Cricket is my favourite game


NameError: name 'xgb_clf' is not defined